# Replicating GkmExplain Analyses

Reproduces the core analyses from:

> Shrikumar A, Prakash E, Kundaje A. GkmExplain: fast and accurate interpretation
> of nonlinear gapped k-mer SVMs. *Bioinformatics* 35(14):i173–i182 (2019).
> [doi:10.1093/bioinformatics/btz322](https://doi.org/10.1093/bioinformatics/btz322)

Uses the **exact train/test splits** from the paper's
[GitHub repository](https://github.com/kundajelab/gkmexplain), including
Nanog ChIP-seq peaks (ENCFF148PBJ) as positives and DNase-seq peaks
(ENCSR000EMU) as GC-matched negatives — the same data used to produce
the paper's figures.

**Pipeline**: download paper data → train gkm-SVM → GkmExplain → ISM comparison → TF-MoDISco

### Requirements

```bash
pip install gkmsvm-lite[examples]
```

In [ ]:
from pathlib import Path

import numpy as np

from gkmsvm.backend import HAS_CUPY, HAS_MLX, to_cpu

# ── Device auto-detection ─────────────────────────────────────────
if HAS_CUPY:
    DEVICE = "cuda"
    print("NVIDIA GPU detected (CuPy) — using CUDA acceleration")
elif HAS_MLX:
    DEVICE = "mlx"
    print("Apple Silicon detected (MLX) — using Metal acceleration")
else:
    DEVICE = "cpu"
    print("No GPU detected — using CPU")

# ── Paths ─────────────────────────────────────────────────────────
REPO_URL = "https://raw.githubusercontent.com/kundajelab/gkmexplain/master/Nanog_H1ESC"
OUTDIR = Path("output_gkmexplain")
OUTDIR.mkdir(exist_ok=True)

# Training parameters (LS-GKM defaults, matching the paper)
L, K, D = 11, 7, 3
C = 1.0

## 1. Data Download

Download the exact train/test FASTAs and pre-trained model from the
[kundajelab/gkmexplain](https://github.com/kundajelab/gkmexplain) repository.

- **Positives**: Nanog ChIP-seq peaks in H1-hESC (ENCFF148PBJ, conservative IDR, hg19)
- **Negatives**: DNase-seq peaks (ENCSR000EMU) — GC-matched accessible regions
  in H1-hESC that lack Nanog binding
- **Pre-trained model**: LS-GKM with default parameters (l=11, k=7, d=3, kernel type 2)</cell id="data-header">


In [ ]:
import subprocess

FILES = {
    "positives_train.fa": "Training positives (Nanog ChIP-seq peaks)",
    "negatives_train.fa": "Training negatives (DNase-seq peaks)",
    "positives_test.fa": "Test positives",
    "negatives_test.fa": "Test negatives",
    "lsgkm_defaultsettings_t2.model.txt": "Pre-trained LS-GKM model",
}

for fname, desc in FILES.items():
    fpath = OUTDIR / fname
    if not fpath.exists():
        print(f"Downloading {fname} ({desc})...")
        subprocess.run([
            "curl", "-sL", "-o", str(fpath),
            f"{REPO_URL}/{fname}",
        ], check=True)
    else:
        print(f"  {fname}")

print("All files downloaded.")

### Load sequences

Load the paper's exact train/test FASTAs and one-hot encode them.

In [ ]:
from gkmsvm import read_fasta, one_hot_encode, one_hot_decode

def load_fasta_ohe(path):
    seqs = [one_hot_encode(seq) for _, seq in read_fasta(str(path))]
    return np.stack(seqs)

X_train_pos = load_fasta_ohe(OUTDIR / "positives_train.fa")
X_train_neg = load_fasta_ohe(OUTDIR / "negatives_train.fa")
X_test_pos = load_fasta_ohe(OUTDIR / "positives_test.fa")
X_test_neg = load_fasta_ohe(OUTDIR / "negatives_test.fa")

print(f"Train positives: {X_train_pos.shape}")
print(f"Train negatives: {X_train_neg.shape}")
print(f"Test positives:  {X_test_pos.shape}")
print(f"Test negatives:  {X_test_neg.shape}")

### Negative sequences

The paper used DNase-seq peaks from [ENCSR000EMU](https://www.encodeproject.org/experiments/ENCSR000EMU/)
as negatives — GC-matched accessible regions in H1-hESC that lack Nanog
binding. These are included in the downloaded FASTAs.

In [ ]:
pos_gc = X_train_pos[:, [1, 2], :].sum() / X_train_pos[:, :, :].sum()
neg_gc = X_train_neg[:, [1, 2], :].sum() / X_train_neg[:, :, :].sum()
print(f"Train positives: {X_train_pos.shape[0]}")
print(f"Train negatives: {X_train_neg.shape[0]}")
print(f"GC content — positives: {pos_gc:.3f}, negatives: {neg_gc:.3f}")

## 2. Train gkm-SVM

Train with default LS-GKM parameters: `estimated` kernel (gkm_esttrunc),
l=11, k=7, d=3. Solver auto-selects between precomputed Gram (small N)
and column-cached SMO (large N).

In [ ]:
from gkmsvm import train_gkmsvm, save_npz
import time

model_path = OUTDIR / "nanog_h1hesc_model.npz"

t0 = time.time()
model = train_gkmsvm(
    X_train_pos, X_train_neg,
    kernel_type="estimated", l=L, k=K, d=D,
    C=C, solver="auto",
    device=DEVICE, verbose=True,
)
elapsed = time.time() - t0
save_npz(model, model_path)
print(f"\nTrained in {elapsed:.1f}s — {model.num_support_vectors} SVs")
print(f"Saved to {model_path}")

if DEVICE == "cuda":
    model.cuda()
    print("Model moved to CUDA for inference")
elif DEVICE == "mlx":
    model.mlx()
    print("Model moved to MLX for inference")

In [ ]:
pos_scores = to_cpu(model(X_test_pos).flatten())
neg_scores = to_cpu(model(X_test_neg).flatten())

print(f"Test positive mean: {pos_scores.mean():.4f} ± {pos_scores.std():.4f}")
print(f"Test negative mean: {neg_scores.mean():.4f} ± {neg_scores.std():.4f}")
print(f"Separation:         {(pos_scores.mean() - neg_scores.mean()):.4f}")

### Compare with the paper's pre-trained model

Load the LS-GKM model from the paper's repository and verify that our
retrained model produces similar scores on the test set.

In [ ]:
from gkmsvm import load_lsgkm_model

pretrained = load_lsgkm_model(str(OUTDIR / "lsgkm_defaultsettings_t2.model.txt"))
if DEVICE == "cuda":
    pretrained.cuda()
elif DEVICE == "mlx":
    pretrained.mlx()

print(f"Pre-trained model: {pretrained.num_support_vectors} SVs, bias={pretrained.bias:.4f}")
print(f"Our model:         {model.num_support_vectors} SVs, bias={model.bias:.4f}")

pre_pos = to_cpu(pretrained(X_test_pos).flatten())
pre_neg = to_cpu(pretrained(X_test_neg).flatten())

corr_pos = np.corrcoef(pos_scores, pre_pos)[0, 1]
corr_neg = np.corrcoef(neg_scores, pre_neg)[0, 1]
print(f"\nScore correlation (test positives): r = {corr_pos:.4f}")
print(f"Score correlation (test negatives): r = {corr_neg:.4f}")
print(f"Pre-trained positive mean: {pre_pos.mean():.4f} ± {pre_pos.std():.4f}")
print(f"Pre-trained negative mean: {pre_neg.mean():.4f} ± {pre_neg.std():.4f}")

## 3. GkmExplain Attribution

GkmExplain analytically decomposes the SVM decision function into per-base
importance scores.

- **mode=0** (importance): contribution of the actual base at each position
- **mode=1** (hypothetical): contribution each base *would have* at each position

Mode 1 is needed for TF-MoDISco — it reveals the full motif pattern including
positions where the actual base is not the most informative.

In [ ]:
from gkmsvm import gkmexplain
import time

X_explain = X_test_pos

t0 = time.time()
attr_m0 = gkmexplain(model, X_explain, mode=0, verbose=True)
time_m0 = time.time() - t0

t0 = time.time()
attr_m1 = gkmexplain(model, X_explain, mode=1, verbose=True)
time_m1 = time.time() - t0

attr_m0 = to_cpu(attr_m0)
attr_m1 = to_cpu(attr_m1)
X_explain_cpu = to_cpu(X_explain)

print(f"\nMode 0 (importance):    {attr_m0.shape}, {time_m0:.1f}s")
print(f"Mode 1 (hypothetical):  {attr_m1.shape}, {time_m1:.1f}s")
print(f"Throughput: {len(X_explain)} seqs in {time_m1:.1f}s ({DEVICE})")

In [ ]:
# Mode 0 only has values at the actual base; mode 1 has values at all 4 bases.
# At the actual base, mode 0 and mode 1 should agree.
actual_m0 = (attr_m0 * X_explain_cpu).sum(axis=1)  # [B, L]
actual_m1 = (attr_m1 * X_explain_cpu).sum(axis=1)  # [B, L]

corr = np.corrcoef(actual_m0.ravel(), actual_m1.ravel())[0, 1]
print(f"Mode 0 vs Mode 1 at actual bases: r = {corr:.6f}")

## 4. ISM Comparison

In-silico mutagenesis (ISM) computes importance by exhaustively substituting
every base at every position. GkmExplain should produce similar attributions
but much faster.

Run on a small subset to compare — ISM is O(3·L) forward passes per sequence.

In [ ]:
from gkmsvm import ism
import time

N_ISM = min(50, len(X_explain))
X_ism = X_explain[:N_ISM]

t0 = time.time()
ism_scores = to_cpu(ism(model, X_ism, verbose=True))
time_ism = time.time() - t0

t0 = time.time()
attr_subset = to_cpu(gkmexplain(model, X_ism, mode=1, verbose=True))
time_explain = time.time() - t0

print(f"\nISM:        {time_ism:.1f}s for {N_ISM} sequences")
print(f"GkmExplain: {time_explain:.1f}s for {N_ISM} sequences")
print(f"Speedup:    {time_ism / max(time_explain, 0.01):.1f}x")

In [ ]:
# Compare ISM vs GkmExplain (mode 1) at non-reference positions.
# ISM delta at the reference base is 0 by construction (no substitution),
# so only non-reference positions carry signal for comparison.
X_ism_cpu = to_cpu(X_ism)
nonref_mask = X_ism_cpu.ravel() == 0

ism_nonref = ism_scores.ravel()[nonref_mask]
explain_nonref = attr_subset.ravel()[nonref_mask]
corr_nonref = np.corrcoef(ism_nonref, explain_nonref)[0, 1]
print(f"ISM vs GkmExplain (non-reference bases): r = {corr_nonref:.4f}")

# Per-position importance: GkmExplain mode 0 vs ISM signed importance
# ISM importance = -(max alt score delta), i.e., how much the best
# mutation improves the score (negative = ref base is important)
attr_m0_sub = to_cpu(gkmexplain(model, X_ism, mode=0, verbose=False))
m0_importance = (attr_m0_sub * X_ism_cpu).sum(axis=1)  # [B, L]
ism_alt_max = (ism_scores * (1 - X_ism_cpu)).max(axis=1)  # [B, L] best alt delta
corr_imp = np.corrcoef(m0_importance.ravel(), ism_alt_max.ravel())[0, 1]
print(f"GkmExplain mode 0 vs ISM max-alt delta:  r = {corr_imp:.4f}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    ax = axes[0]
    subsample = np.random.default_rng(0).choice(len(ism_nonref), size=min(5000, len(ism_nonref)), replace=False)
    ax.scatter(ism_nonref[subsample], explain_nonref[subsample], alpha=0.2, s=5, rasterized=True)
    ax.set_xlabel("ISM delta")
    ax.set_ylabel("GkmExplain hypothetical")
    ax.set_title(f"Non-reference bases (r={corr_nonref:.3f})")

    ax = axes[1]
    ax.scatter(ism_alt_max.ravel(), m0_importance.ravel(), alpha=0.3, s=5, rasterized=True)
    ax.set_xlabel("ISM max-alt delta")
    ax.set_ylabel("GkmExplain mode 0 importance")
    ax.set_title(f"Per-position importance (r={corr_imp:.3f})")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not installed, skipping plot)")

## 5. Visualization

Plot per-position GkmExplain importance for a few example sequences.
High-scoring sequences should show clear peaks at Nanog/Oct4/Sox2
motif positions.

In [ ]:
test_scores = to_cpu(model(X_explain).flatten())
top_idx = np.argsort(test_scores)[::-1][:5]

for rank, idx in enumerate(top_idx):
    print(f"  #{rank+1}  score={test_scores[idx]:.4f}  idx={idx}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    n_show = min(3, len(top_idx))
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]

    for ax, idx in zip(axes, top_idx[:n_show]):
        imp = (attr_m1[idx] * X_explain_cpu[idx]).sum(axis=0)  # [L]
        colors = np.where(imp > 0, "#2166ac", "#b2182b")
        ax.bar(range(len(imp)), imp, color=colors, width=1.0, linewidth=0)
        ax.set_ylabel("Importance")
        ax.set_title(f"Sequence {idx} (score={test_scores[idx]:.3f})", fontsize=10)
        ax.axhline(0, color="grey", linewidth=0.5)

    axes[-1].set_xlabel("Position")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not installed, skipping plot)")

In [ ]:
try:
    import matplotlib.pyplot as plt

    # Show full [4, L] attribution heatmap for the top sequence
    idx = top_idx[0]
    fig, ax = plt.subplots(figsize=(14, 2))
    im = ax.imshow(attr_m1[idx], aspect="auto", cmap="RdBu_r",
                   vmin=-np.abs(attr_m1[idx]).max(),
                   vmax=np.abs(attr_m1[idx]).max())
    ax.set_yticks(range(4))
    ax.set_yticklabels(["A", "C", "G", "T"])
    ax.set_xlabel("Position")
    ax.set_title(f"GkmExplain hypothetical importance — sequence {idx}")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Attribution")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not installed, skipping plot)")

## 6. TF-MoDISco Motif Discovery

Export GkmExplain attributions (mode=1, hypothetical) and one-hot sequences
in the format expected by [TF-MoDISco](https://github.com/jmschrei/tfmodisco-lite).

TF-MoDISco clusters recurring attribution patterns to discover motifs.
For Nanog, it should recover the canonical Oct4-Sox2-Nanog composite motif
and associated co-factor binding sites — matching the motifs shown in
the GkmExplain paper (Figure 4).

In [ ]:
# Free GPU memory — all remaining work (MoDISco, plots) is CPU-only
if DEVICE != "cpu":
    model.cpu()
    pretrained.cpu()
    print("Models moved to CPU, GPU memory released")

In [ ]:
attr_path = OUTDIR / "nanog_attr.npz"
seq_path = OUTDIR / "nanog_seqs.npz"

np.savez(attr_path, attr_m1.astype(np.float32))
np.savez(seq_path, X_explain_cpu.astype(np.float32))

print(f"Attributions: {attr_path}  shape={attr_m1.shape}")
print(f"Sequences:    {seq_path}  shape={X_explain_cpu.shape}")
print()
print("To run TF-MoDISco from the command line:")
print(f"  modisco motifs -s {seq_path} -a {attr_path} -n 2000 -o {OUTDIR}/modisco_results.h5")

In [ ]:
# Run TF-MoDISco programmatically (if installed)
modisco_results_path = OUTDIR / "modisco_results.h5"

try:
    import modiscolite

    if not modisco_results_path.exists():
        # TF-MoDISco expects [B, L, 4] (channels-last)
        seqs_cl = X_explain_cpu.astype(np.float32).transpose(0, 2, 1)  # [B, L, 4]
        attr_cl = attr_m1.astype(np.float32).transpose(0, 2, 1)        # [B, L, 4]

        pos_patterns, neg_patterns = modiscolite.tfmodisco.TFMoDISco(
            seqs_cl, attr_cl,
            max_seqlets_per_metacluster=2000,
            verbose=True,
        )

        import h5py
        with h5py.File(modisco_results_path, "w") as f:
            if pos_patterns is not None:
                for i, pattern in enumerate(pos_patterns):
                    grp = f.create_group(f"pos_patterns/pattern_{i}")
                    grp.create_dataset("contrib_scores", data=pattern.contrib_scores)
                    grp.create_dataset("hypothetical_contribs", data=pattern.hypothetical_contribs)
                    grp.create_dataset("sequence", data=pattern.sequence)
            if neg_patterns is not None:
                for i, pattern in enumerate(neg_patterns):
                    grp = f.create_group(f"neg_patterns/pattern_{i}")
                    grp.create_dataset("contrib_scores", data=pattern.contrib_scores)
                    grp.create_dataset("hypothetical_contribs", data=pattern.hypothetical_contribs)
                    grp.create_dataset("sequence", data=pattern.sequence)

        n_pos = len(pos_patterns) if pos_patterns is not None else 0
        n_neg = len(neg_patterns) if neg_patterns is not None else 0
        print(f"Found {n_pos} positive, {n_neg} negative patterns")
        print(f"Saved to {modisco_results_path}")
    else:
        print(f"Results already exist: {modisco_results_path}")

except ImportError:
    print("modisco not installed. Install with: pip install modisco")
    print(f"Then run: modisco motifs -s {seq_path} -a {attr_path} -n 2000 -o {modisco_results_path}")

In [ ]:
# Visualize discovered motifs as sequence logos
try:
    import h5py
    import matplotlib.pyplot as plt
    import logomaker
    import pandas as pd

    if modisco_results_path.exists():
        with h5py.File(modisco_results_path, "r") as f:
            patterns = []
            if "pos_patterns" in f:
                for key in sorted(f["pos_patterns"].keys()):
                    cwm = f[f"pos_patterns/{key}/contrib_scores"][:]  # [L, 4]
                    patterns.append((key, cwm))

        n_show = min(4, len(patterns))
        if n_show > 0:
            fig, axes = plt.subplots(n_show, 1, figsize=(10, 2 * n_show))
            if n_show == 1:
                axes = [axes]

            for ax, (name, cwm) in zip(axes, patterns[:n_show]):
                df = pd.DataFrame(cwm, columns=["A", "C", "G", "T"])
                logomaker.Logo(df, ax=ax, color_scheme="classic")
                ax.set_title(name, fontsize=10)
                ax.set_ylabel("CWM")

            axes[-1].set_xlabel("Position")
            plt.suptitle("TF-MoDISco motifs from GkmExplain attributions", y=1.02)
            plt.tight_layout()
            plt.show()
        else:
            print("No positive patterns found.")
    else:
        print(f"Run TF-MoDISco first to generate {modisco_results_path}")

except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: pip install logomaker h5py matplotlib")

## Summary

This notebook replicated the GkmExplain pipeline from Shrikumar et al. (2019)
using the **exact data** from the paper's
[GitHub repository](https://github.com/kundajelab/gkmexplain):

1. **Trained** a gkm-SVM on the paper's exact Nanog H1-hESC train/test splits
   (ChIP-seq positives + DNase-seq negatives)
2. **Compared** our retrained model with the paper's pre-trained LS-GKM model
3. **GkmExplain** computed per-position attributions (mode 0 + mode 1) for test peaks
4. **ISM comparison** showed GkmExplain correlates with ISM but runs faster
5. **TF-MoDISco** discovered motifs from GkmExplain attributions

### Data sources

| Dataset | Source | Description |
|---------|--------|-------------|
| Training positives | [positives_train.fa](https://github.com/kundajelab/gkmexplain/blob/master/Nanog_H1ESC/positives_train.fa) | 5,647 Nanog ChIP-seq peaks (ENCFF148PBJ, hg19) |
| Training negatives | [negatives_train.fa](https://github.com/kundajelab/gkmexplain/blob/master/Nanog_H1ESC/negatives_train.fa) | 5,981 DNase-seq peaks (ENCSR000EMU, hg19) |
| Test positives | [positives_test.fa](https://github.com/kundajelab/gkmexplain/blob/master/Nanog_H1ESC/positives_test.fa) | Held-out Nanog peaks |
| Test negatives | [negatives_test.fa](https://github.com/kundajelab/gkmexplain/blob/master/Nanog_H1ESC/negatives_test.fa) | Held-out DNase peaks |
| Pre-trained model | [lsgkm_defaultsettings_t2.model.txt](https://github.com/kundajelab/gkmexplain/blob/master/Nanog_H1ESC/lsgkm_defaultsettings_t2.model.txt) | LS-GKM, l=11 k=7 d=3 |

### CLI equivalent

```bash
gkmsvm -v train -p positives_train.fa -n negatives_train.fa -o model.npz
gkmsvm -v explain -m model.npz -i positives_test.fa -o attr.npz -s seqs.npz
modisco motifs -s seqs.npz -a attr.npz -n 2000 -o modisco.h5
```